In [61]:
import os
import pandas as pd
import numpy as np

## Train Metadata

In [34]:
base_dir = r"C:\\Users\\USER\\.cache\\huggingface\\hub\datasets--nexar-ai--nexar_collision_prediction\\snapshots\\aa97deda5a59f00bb7187739053b7c72e14374df\\train"

pos_dir = os.path.join(base_dir, "positive")
neg_dir = os.path.join(base_dir, "negative")

pos_meta = pd.read_csv(os.path.join(pos_dir, "metadata.csv"))
neg_meta = pd.read_csv(os.path.join(neg_dir, "metadata.csv"))

meta = pd.concat([pos_meta.assign(folder=pos_dir), neg_meta.assign(folder=neg_dir)], ignore_index=True)
meta['label'] = meta['folder'].apply(lambda f: 1 if 'positive' in f.lower() else 0).astype('int8')

print("Total videos:", len(meta))
meta.info()

Total videos: 1500
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   file_name         1500 non-null   object 
 1   time_of_event     750 non-null    float64
 2   time_of_alert     750 non-null    float64
 3   light_conditions  1500 non-null   object 
 4   weather           1498 non-null   object 
 5   scene             1500 non-null   object 
 6   time_to_accident  0 non-null      float64
 7   folder            1500 non-null   object 
 8   label             1500 non-null   int8   
dtypes: float64(3), int8(1), object(5)
memory usage: 95.3+ KB


In [35]:
print(meta.light_conditions.value_counts())
print(meta.weather.value_counts())
print(meta.scene.value_counts())
print(meta.label.value_counts())

light_conditions
Normal      1355
Twilight      88
Dark          47
Bright        10
Name: count, dtype: int64
weather
Clear     919
Cloudy    495
Rain       83
Snow        1
Name: count, dtype: int64
scene
Urban         784
Highway       379
Sub-urban     271
Other          34
Rural          18
Industrial     13
Nature          1
Name: count, dtype: int64
label
1    750
0    750
Name: count, dtype: int64


### light_conditions

In [36]:
meta['light_conditions'] = meta['light_conditions'].replace({
    'Bright': 'Normal',
    'Twilight': 'LowLight',
    'Dark': 'LowLight'})

meta['light_conditions'].value_counts()

light_conditions
Normal      1365
LowLight     135
Name: count, dtype: int64

### weather

In [37]:
meta['weather'] = meta['weather'].replace({'Snow': 'Rain'})
meta['weather'].value_counts()

weather
Clear     919
Cloudy    495
Rain       84
Name: count, dtype: int64

### scene

In [38]:
meta['scene'] = meta['scene'].replace({
    'Sub-urban': 'Urban',
    'Rural': 'Rural',
    'Industrial': 'Rural',
    'Nature': 'Rural',
    'Other': 'Rural'})

meta['scene'].value_counts()

scene
Urban      1055
Highway     379
Rural        66
Name: count, dtype: int64

### Encode

In [39]:
print(meta.light_conditions.value_counts())
print(meta.weather.value_counts())
print(meta.scene.value_counts())

light_conditions
Normal      1365
LowLight     135
Name: count, dtype: int64
weather
Clear     919
Cloudy    495
Rain       84
Name: count, dtype: int64
scene
Urban      1055
Highway     379
Rural        66
Name: count, dtype: int64


In [40]:
meta['light_conditions'] = meta['light_conditions'].map({
    'Normal': 0, 'LowLight': 1}).astype('int8')

def encode_weather(w):
    if w == "Clear":
        return pd.Series([0, 0])
    elif w == "Cloudy":
        return pd.Series([1, 0])
    else:  # Rain (includes Snow)
        return pd.Series([0, 1])

meta[['weather_cloudy', 'weather_rain']] = meta['weather'].apply(encode_weather).astype('int8')

def encode_scene(s):
    if s == "Urban":
        return pd.Series([0, 0])
    elif s == "Highway":
        return pd.Series([1, 0])
    else:  # Rural
        return pd.Series([0, 1])

meta[['scene_highway', 'scene_rural']] = meta['scene'].apply(encode_scene).astype('int8')

In [ ]:
meta.drop(columns=['weather', 'scene', 'time_to_accident'], inplace=True)
meta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   file_name         1500 non-null   object 
 1   time_of_event     750 non-null    float64
 2   time_of_alert     750 non-null    float64
 3   light_conditions  1500 non-null   int8   
 4   folder            1500 non-null   object 
 5   label             1500 non-null   int8   
 6   weather_cloudy    1500 non-null   int8   
 7   weather_rain      1500 non-null   int8   
 8   scene_highway     1500 non-null   int8   
 9   scene_rural       1500 non-null   int8   
dtypes: float64(2), int8(6), object(2)
memory usage: 55.8+ KB


## Test Metadata

In [53]:
test_dir = r"C:\\Users\\USER\\.cache\\huggingface\\hub\\datasets--nexar-ai--nexar_collision_prediction\\snapshots\\aa97deda5a59f00bb7187739053b7c72e14374df\\test-public"

pos_dir_tst = os.path.join(test_dir, "positive")
neg_dir_tst = os.path.join(test_dir, "negative")

pos_meta_tst = pd.read_csv(os.path.join(pos_dir_tst, "metadata.csv"))
neg_meta_tst = pd.read_csv(os.path.join(neg_dir_tst, "metadata.csv"))


test_dir_pr = r"C:\\Users\\USER\\.cache\\huggingface\\hub\\datasets--nexar-ai--nexar_collision_prediction\\snapshots\\aa97deda5a59f00bb7187739053b7c72e14374df\\test-private"

pos_dir_tst_pr = os.path.join(test_dir_pr, "positive")
neg_dir_tst_pr = os.path.join(test_dir_pr, "negative")

pos_meta_tst_pr = pd.read_csv(os.path.join(pos_dir_tst_pr, "metadata.csv"))
neg_meta_tst_pr = pd.read_csv(os.path.join(neg_dir_tst_pr, "metadata.csv"))

meta_test = pd.concat([pos_meta_tst.assign(folder=pos_dir_tst), neg_meta_tst.assign(folder=neg_dir_tst), 
                       pos_meta_tst_pr.assign(folder=pos_dir_tst_pr), neg_meta_tst_pr.assign(folder=neg_dir_tst_pr)], ignore_index=True)

meta_test['label'] = meta_test['folder'].apply(lambda f: 1 if 'positive' in f.lower() else 0).astype('int8')

print("Total test videos:", len(meta_test))
meta_test.info()

Total test videos: 1344
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1344 entries, 0 to 1343
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   file_name         1344 non-null   object 
 1   time_of_event     672 non-null    float64
 2   time_of_alert     672 non-null    float64
 3   light_conditions  1344 non-null   object 
 4   weather           1344 non-null   object 
 5   scene             1344 non-null   object 
 6   time_to_accident  1344 non-null   float64
 7   folder            1344 non-null   object 
 8   label             1344 non-null   int8   
dtypes: float64(3), int8(1), object(5)
memory usage: 85.4+ KB


In [54]:
print(meta_test.light_conditions.value_counts())
print(meta_test.weather.value_counts())
print(meta_test.scene.value_counts())
print(meta_test.label.value_counts())

light_conditions
Normal      1245
Twilight      65
Dark          26
Bright         8
Name: count, dtype: int64
weather
Clear     873
Cloudy    399
Rain       69
Fog         3
Name: count, dtype: int64
scene
Urban         690
Highway       344
Sub-urban     227
Other          37
Rural          32
Industrial     14
Name: count, dtype: int64
label
1    672
0    672
Name: count, dtype: int64


### light_conditions

In [55]:
meta_test['light_conditions'] = meta_test['light_conditions'].replace({
    'Bright': 'Normal',
    'Twilight': 'LowLight',
    'Dark': 'LowLight'})

meta_test['light_conditions'].value_counts()

light_conditions
Normal      1253
LowLight      91
Name: count, dtype: int64

### weather

In [56]:
meta_test['weather'] = meta_test['weather'].replace({'Fog': 'Rain'})
meta_test['weather'].value_counts()

weather
Clear     873
Cloudy    399
Rain       72
Name: count, dtype: int64

### scene

In [57]:
meta_test['scene'] = meta_test['scene'].replace({
    'Sub-urban': 'Urban',
    'Rural': 'Rural',
    'Industrial': 'Rural',
    'Nature': 'Rural',
    'Other': 'Rural'})

meta_test['scene'].value_counts()

scene
Urban      917
Highway    344
Rural       83
Name: count, dtype: int64

### Encode

In [58]:
print(meta_test.light_conditions.value_counts())
print(meta_test.weather.value_counts())
print(meta_test.scene.value_counts())

light_conditions
Normal      1253
LowLight      91
Name: count, dtype: int64
weather
Clear     873
Cloudy    399
Rain       72
Name: count, dtype: int64
scene
Urban      917
Highway    344
Rural       83
Name: count, dtype: int64


In [59]:
meta_test['light_conditions'] = meta_test['light_conditions'].map({
    'Normal': 0, 'LowLight': 1}).astype('int8')

def encode_weather(w):
    if w == "Clear":
        return pd.Series([0, 0])
    elif w == "Cloudy":
        return pd.Series([1, 0])
    else:  # Rain (includes Snow)
        return pd.Series([0, 1])

meta_test[['weather_cloudy', 'weather_rain']] = meta_test['weather'].apply(encode_weather).astype('int8')

def encode_scene(s):
    if s == "Urban":
        return pd.Series([0, 0])
    elif s == "Highway":
        return pd.Series([1, 0])
    else:  # Rural
        return pd.Series([0, 1])

meta_test[['scene_highway', 'scene_rural']] = meta_test['scene'].apply(encode_scene).astype('int8')

In [60]:
meta_test.drop(columns=['weather', 'scene', 'time_to_accident'], inplace=True)
meta_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1344 entries, 0 to 1343
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   file_name         1344 non-null   object 
 1   time_of_event     672 non-null    float64
 2   time_of_alert     672 non-null    float64
 3   light_conditions  1344 non-null   int8   
 4   folder            1344 non-null   object 
 5   label             1344 non-null   int8   
 6   weather_cloudy    1344 non-null   int8   
 7   weather_rain      1344 non-null   int8   
 8   scene_highway     1344 non-null   int8   
 9   scene_rural       1344 non-null   int8   
dtypes: float64(2), int8(6), object(2)
memory usage: 50.0+ KB


In [62]:
meta.to_pickle("train_meta_processed.pkl")
meta_test.to_pickle("test_meta_processed.pkl")